# Modelo GCN + LSTM — grafo vial (Iztapalapa)

Topología: red vial (OSMnx), justificada empíricamente en `moran_grafo_vial.ipynb` / `h3_2.ipynb` (autocorrelación espacial positiva y significativa solo con adyacencia de red vial real, no con malla H3). Ventana temporal: 2021 en adelante, misma que la validación de Moran's I.

In [1]:
import numpy as np
import pandas as pd
import networkx as nx
import osmnx as ox
import torch
import torch.nn as nn
from torch_geometric.nn import GCNConv
from scipy.sparse import coo_matrix

pd.set_option('display.max_columns', None)


d:\Lic Ciencia de Datos\TT C5\venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


## 1. Carga y filtrado

Se conservan filas con `target == 0` (a diferencia del filtro usado para Moran's I): aquí `target` se necesita completo para sumar correctamente los incidentes confirmados por (nodo, hora) — una fila con `target == 0` simplemente no le suma nada a esa celda-hora, que es el comportamiento correcto para construir el panel.

In [2]:
def filtrar_dataset(df: pd.DataFrame, anio_inicio: int) -> pd.DataFrame:
    """
    Ventana temporal + filtros de calidad para el panel de entrenamiento.

    A diferencia del filtro usado en la validación de Moran's I (que
    excluía target == 0 porque esa prueba mide autocorrelación sobre
    conteos TOTALES de incidentes confirmados), aquí se conservan todas
    las filas -- incluyendo target == 0 (falsas alarmas) -- porque
    'target' se necesita completo para sumar correctamente los
    incidentes confirmados por (nodo, hora) en el siguiente paso.
    """
    mask = (
        (df['anio'] >= anio_inicio)
        & (df['coord_valida'] == True)
        & (df['es_duplicado_c5'] == False)
    )
    return df.loc[mask].copy()


df = pd.read_csv('df_hour_clean.csv')
df_filtrado = filtrar_dataset(df, anio_inicio=2021)
print(f"Registros tras filtrado: {len(df_filtrado):,} de {len(df):,} originales "
      f"({len(df_filtrado) / len(df) * 100:.1f}%)")


Registros tras filtrado: 66,320 de 145,313 originales (45.6%)


## 2. Grafo vial y `edge_index`

Misma receta ya validada en `moran_grafo_vial.ipynb`: grafo no dirigido, aristas deduplicadas. **Requiere red** (Nominatim/Overpass) — correr en el venv local, no en un sandbox aislado.

In [3]:
def obtener_grafo_vial() -> nx.MultiDiGraph:
    """
    Descarga la red vial de Iztapalapa vía OSMnx.
    network_type='drive' porque el fenómeno es vehicular -- no tiene
    sentido incluir andadores peatonales en la topología de propagación
    del riesgo.
    """
    gdf_limite = ox.geocode_to_gdf("Iztapalapa, Ciudad de México, México")
    return ox.graph_from_polygon(gdf_limite.geometry.union_all(), network_type='drive', simplify=True)


def construir_edge_index(G: nx.MultiDiGraph, nodo_a_idx: dict) -> torch.Tensor:
    """
    edge_index no dirigido, sin aristas duplicadas, en el mismo orden de
    índices que nodo_a_idx (debe ser el MISMO diccionario usado para
    construir la matriz del panel en la sección 3 -- si difiere, los
    índices de nodo no coinciden entre el grafo y el panel).
    """
    G_ud = G.to_undirected()
    aristas = set()
    for origen, destino in G_ud.edges():
        i, j = nodo_a_idx[origen], nodo_a_idx[destino]
        aristas.add((i, j))
        aristas.add((j, i))

    return torch.tensor(list(aristas), dtype=torch.long).t().contiguous()


G_vial = obtener_grafo_vial()
nodos_unicos = list(G_vial.nodes())
nodo_a_idx = {nodo: i for i, nodo in enumerate(nodos_unicos)}

edge_index = construir_edge_index(G_vial, nodo_a_idx)
print(f"Nodos: {len(nodos_unicos):,} | Aristas (no dirigido, dedup): {edge_index.shape[1]:,}")


Nodos: 22,875 | Aristas (no dirigido, dedup): 63,489


## 3. Asignación de `nodo_id` y panel disperso (horas × nodos)

Se construye directo en formato disperso (CSR): las combinaciones (nodo, hora) sin ninguna llamada quedan en cero **implícitamente**, sin materializarse como filas -- no hace falta expandir un panel denso de miles de millones de celdas.

In [5]:
df_filtrado['fecha_hora_rango'] = pd.to_datetime(df_filtrado['fecha_hora_rango'])

In [6]:
def asignar_nodo_vial(df: pd.DataFrame, G: nx.MultiDiGraph) -> pd.DataFrame:
    """Asigna a cada incidente el nodo (intersección) más cercano de la red vial."""
    df = df.copy()
    df['nodo_id'] = ox.distance.nearest_nodes(G, X=df['longitud'].values, Y=df['latitud'].values)
    return df


def construir_panel_disperso(df: pd.DataFrame, nodo_a_idx: dict, freq: str = 'h'):
    """
    Construye el panel completo (horas x nodos) en formato disperso CSR,
    con el conteo de incidentes CONFIRMADOS (target == 1) por (nodo, hora).

    Retorna (matriz_csr, horas_completas, hora_a_idx).
    """
    horas_completas = pd.date_range(
        start=df['fecha_hora_rango'].min(),
        end=df['fecha_hora_rango'].max(),
        freq=freq,
    )
    hora_a_idx = {hora: i for i, hora in enumerate(horas_completas)}

    incidentes_por_grupo = df.groupby(['nodo_id', 'fecha_hora_rango'])['target'].sum()
    incidentes_por_grupo = incidentes_por_grupo[incidentes_por_grupo > 0]  # solo entradas no-cero

    filas = [hora_a_idx[h] for (_, h) in incidentes_por_grupo.index]
    columnas = [nodo_a_idx[n] for (n, _) in incidentes_por_grupo.index]
    valores = incidentes_por_grupo.values

    matriz = coo_matrix(
        (valores, (filas, columnas)),
        shape=(len(horas_completas), len(nodo_a_idx)),
    ).tocsr()

    return matriz, horas_completas, hora_a_idx


df_filtrado = asignar_nodo_vial(df_filtrado, G_vial)
matriz_csr, horas_completas, hora_a_idx = construir_panel_disperso(df_filtrado, nodo_a_idx)

print(f"Panel: {matriz_csr.shape[0]:,} horas x {matriz_csr.shape[1]:,} nodos")
print(f"Suma total de incidentes en el panel : {matriz_csr.sum():,.0f}")
print(f"Suma total de target en df_filtrado   : {df_filtrado['target'].sum():,}")
assert matriz_csr.sum() == df_filtrado['target'].sum(), "el panel no cuadra con el total de incidentes confirmados"


Panel: 27,716 horas x 22,875 nodos
Suma total de incidentes en el panel : 38,697
Suma total de target en df_filtrado   : 38,697


## 4. Ventana deslizante

Corta directo sobre la matriz dispersa (sin densificar el panel completo).

In [7]:
def obtener_ventana(matriz_csr, inicio: int, T: int):
    """
    Extrae una ventana de T horas consecutivas (entrada) y la etiqueta
    binaria de la hora siguiente, sin densificar la matriz completa.

    Retorna:
        ventana_entrada : np.ndarray (T, N_nodos) -- conteos de incidentes
        etiqueta        : np.ndarray (N_nodos,)   -- binaria, 1 si hubo
                          al menos un incidente confirmado en la hora siguiente
    """
    ventana_entrada = matriz_csr[inicio: inicio + T, :].toarray()
    incidentes_siguiente = matriz_csr[inicio + T, :].toarray().flatten()
    etiqueta = (incidentes_siguiente > 0).astype(np.float32)
    return ventana_entrada, etiqueta


T = 24  # un día completo de contexto
n_horas_totales = matriz_csr.shape[0]
n_ventanas = n_horas_totales - T
print(f"Ventanas disponibles: {n_ventanas:,}")


Ventanas disponibles: 27,692


## 5. Modelo GCN + LSTM

Patrón B de la comparativa de arquitecturas (encoder secuencial): el GCN se aplica en cada paso de tiempo sobre la MISMA topología (grafo vial, estática), con pesos compartidos entre pasos; la secuencia resultante de embeddings espaciales alimenta al LSTM.

In [8]:
class ModeloGCNLSTM(nn.Module):
    """
    Para cada paso t: embedding_t = GCN(edge_index, X_t)  (pesos compartidos entre pasos)
    Luego: [embedding_1, ..., embedding_T] -> LSTM -> embedding final -> capa de salida.
    """

    def __init__(self, n_features: int, gcn_hidden: int = 32, lstm_hidden: int = 64):
        super().__init__()
        self.gcn = GCNConv(n_features, gcn_hidden)
        self.lstm = nn.LSTM(input_size=gcn_hidden, hidden_size=lstm_hidden, batch_first=True)
        self.salida = nn.Linear(lstm_hidden, 1)  # logit crudo -- BCEWithLogitsLoss aplica el sigmoide

    def forward(self, ventana_entrada: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        """ventana_entrada: (T, N_nodos, n_features)"""
        embeddings_por_hora = []
        for t in range(ventana_entrada.shape[0]):
            x_t = ventana_entrada[t]                          # (N_nodos, n_features)
            emb_t = torch.relu(self.gcn(x_t, edge_index))     # (N_nodos, gcn_hidden)
            embeddings_por_hora.append(emb_t)

        secuencia = torch.stack(embeddings_por_hora, dim=1)   # (N_nodos, T, gcn_hidden)
        _, (h_final, _) = self.lstm(secuencia)
        return self.salida(h_final[-1])                         # (N_nodos, 1)


## 6. Split cronológico (sin fuga de datos)

In [9]:
n_train = int(0.8 * n_ventanas)
n_val = int(0.1 * n_ventanas)

indices_train = list(range(0, n_train))
indices_val = list(range(n_train, n_train + n_val))
indices_test = list(range(n_train + n_val, n_ventanas))

print(f"Train: {len(indices_train):,} | Val: {len(indices_val):,} | Test: {len(indices_test):,}")


Train: 22,153 | Val: 2,769 | Test: 2,770


## 7. Función de pérdida

Clasificación binaria (¿hubo incidente confirmado en la hora siguiente?), con `pos_weight` calculado sobre el panel completo -- incluyendo los ceros implícitos, no solo las celda-hora con alguna llamada registrada.

In [10]:
n_celdas_totales = matriz_csr.shape[0] * matriz_csr.shape[1]
n_positivos = int(matriz_csr.count_nonzero())   # celda-hora con >=1 incidente confirmado
n_negativos = n_celdas_totales - n_positivos      # el resto, incluyendo ceros implícitos

pos_weight = torch.tensor(n_negativos / n_positivos, dtype=torch.float32)
criterio = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print(f"Celda-hora positivas : {n_positivos:,} ({100*n_positivos/n_celdas_totales:.3f}% del panel)")
print(f"Celda-hora negativas : {n_negativos:,}")
print(f"pos_weight           : {pos_weight.item():.2f}")


Celda-hora positivas : 38,348 (0.006% del panel)
Celda-hora negativas : 633,965,152
pos_weight           : 16531.90


## 8. Entrenamiento

Las ventanas consecutivas se traslapan en (T-1)/T de su contenido -- entrenar con TODAS es computacionalmente muy caro (T pasadas de GCN por ventana) y mayormente redundante. Se muestrea con un paso fijo; ajustar `paso` según el tiempo de cómputo disponible.

In [ ]:
modelo = ModeloGCNLSTM(n_features=1)
optimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3)

paso = 6  # una ventana cada 6 horas, en vez de cada 1 -- ajustar según cómputo disponible
indices_train_muestreados = indices_train[::paso]
indices_val_muestreados = indices_val[::paso]

n_epocas = 5

for epoca in range(n_epocas):
    modelo.train()
    perdida_train = 0.0
    for inicio in indices_train_muestreados:
        entrada_np, etiqueta_np = obtener_ventana(matriz_csr, inicio, T)
        entrada = torch.tensor(entrada_np, dtype=torch.float32).unsqueeze(-1)     # (T, N_nodos, 1)
        etiqueta = torch.tensor(etiqueta_np, dtype=torch.float32).unsqueeze(-1)   # (N_nodos, 1)

        optimizador.zero_grad()
        prediccion = modelo(entrada, edge_index)
        perdida = criterio(prediccion, etiqueta)
        perdida.backward()
        optimizador.step()

        perdida_train += perdida.item()

    modelo.eval()
    perdida_val = 0.0
    with torch.no_grad():
        for inicio in indices_val_muestreados:
            entrada_np, etiqueta_np = obtener_ventana(matriz_csr, inicio, T)
            entrada = torch.tensor(entrada_np, dtype=torch.float32).unsqueeze(-1)
            etiqueta = torch.tensor(etiqueta_np, dtype=torch.float32).unsqueeze(-1)
            prediccion = modelo(entrada, edge_index)
            perdida_val += criterio(prediccion, etiqueta).item()

    print(f"Época {epoca + 1}/{n_epocas} | "
          f"pérdida train = {perdida_train / len(indices_train_muestreados):.4f} | "
          f"pérdida val = {perdida_val / len(indices_val_muestreados):.4f}")
